# Live lineup card (Supabase odds → slots → Log5 sketch → `nb_k_v1`)

Live strikeout card from the **latest** snapshot in `odds.mlb_novig`,
`odds.mlb_prophetx`, `odds.mlb_pinnacle`, `odds.mlb_fanduel`,
`odds.mlb_draftkings`, `odds.mlb_underdogs`, and `odds.mlb_prizepicks`
(credentials from `.env`). Lineup slots are **not**
wired into the GLM; the Log5 walk is a notebook sketch after a 100%
`expected_bf` mixture placeholder.

**Phase 1** (network, skip if cached): trailing-365 PBP + today's live nines.
**Phase 2:** match pitcher K quotes, show the nine, Log5 E[K], `nb_k_v1`.
Set `ODDS_SOURCE = "json"` to use local `data/odds/*_mlb_*_props.json` instead.


## Knobs

- First PBP pass is ~1s/game. `PBP_LIMIT` caps **new** fetches for a smoke test.
- Default odds source is Supabase (`ODDS_SOURCE = "supabase"`). Use `"json"` for a local glob.


In [45]:
from __future__ import annotations

from pathlib import Path

ROOT = Path("/Users/alexgonzalez/Documents/nba_quant")
ODDS_SOURCE = "supabase"  # "json" uses ODDS_GLOB
ODDS_GLOB = "data/odds/*_mlb_*_props.json"
PBP_LOOKBACK_DAYS = 365
PBP_LIMIT = None  # e.g. 40 for a smoke run; None = all missing games
FETCH = True  # False = never hit Stats API (local gzip / tables only)


In [46]:
import json
import sys
import warnings
from datetime import UTC, datetime, timedelta

import numpy as np
import pandas as pd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.mlb.config import load_config
from src.mlb.evaluation.backtest import _evaluation_panel, _model_feature_rows
from src.mlb.markets.odds import american_to_implied, expected_profit, implied_to_decimal
from src.mlb.models.batter_rates import is_strikeout, rate_version
from src.mlb.models.serialize import as_strikeout_model, load_model
from src.mlb.models.strikeouts import predict_strikeout_pmf
from src.mlb.pipeline.gamelogs import (
    TRAIN_SEASONS,
    attach_schedule_times,
    default_http,
    load_or_fetch_people,
    load_or_fetch_schedule,
    load_or_fetch_starter_logs,
    normalize_player_name,
)
from src.mlb.pipeline.http import HttpClient
from src.mlb.pipeline.ingest import SCHEDULE_URL, snapshot_raw
from src.mlb.pipeline.lineup_slots import ingest_lineup_slots, select_lineup_slots
from src.mlb.pipeline.parse import parse_schedule
from src.mlb.pipeline.pbp import ingest_play_by_play
from src.mlb.pipeline.hf_tables import pregame_from_starts
from src.mlb.schemas import GAME_VERSION_COLUMNS, PITCHER_START_COLUMNS, PMF_COLUMNS, coerce_frame
from src.mlb.storage import MlbStore

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

config = load_config(ROOT / "config" / "mlb.yaml")
store = MlbStore(config)
http = HttpClient(config) if FETCH else None
artifact = ROOT / "artifacts" / "mlb" / "strikeouts" / f"{config.model_version}.pkl"
model = as_strikeout_model(load_model(artifact))
print("model", artifact)
print("rate_version", rate_version(config))


model /Users/alexgonzalez/Documents/nba_quant/artifacts/mlb/strikeouts/nb_k_v1.pkl
rate_version kpa_52efe66d1aaf


## Odds (latest Supabase snapshot per book)


In [47]:
import os

from sqlalchemy import create_engine, text

ODDS_TABLES = (
    ("novig", "mlb_novig", "stat_name = 'strikeouts'", "sides"),
    ("prophetx", "mlb_prophetx", "stat_name = 'strikeouts'", "sides"),
    ("pinnacle", "mlb_pinnacle", "market_type = 'player_strikeouts'", "sides"),
    ("fanduel", "mlb_fanduel", "market_type = 'player_strikeouts'", "sides"),
    ("draftkings", "mlb_draftkings", "market_type = 'player_strikeouts'", "sides"),
    ("underdog", "mlb_underdogs", "stat_name = 'strikeouts'", "sides"),
    ("prizepicks", "mlb_prizepicks", "stat_type = 'Pitcher Strikeouts'", "dfs"),
)


def _load_env(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("'").strip('"'))


def _finish_quotes(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    out = frame.copy()
    out["scheduled"] = pd.to_datetime(out["scheduled"], utc=True, errors="coerce")
    out["game_date"] = out["scheduled"].dt.strftime("%Y-%m-%d")
    out["player_key"] = out["player"].map(normalize_player_name)
    out["home_key"] = out["home_name"].fillna("").map(normalize_player_name)
    out["away_key"] = out["away_name"].fillna("").map(normalize_player_name)
    return out


def load_prop_files(root: Path, pattern: str) -> pd.DataFrame:
    rows: list[dict] = []
    paths = sorted(root.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"no odds files matching {pattern}")
    for path in paths:
        payload = json.loads(path.read_text())
        book = str(payload.get("source") or path.stem.split("_")[0])
        fetched = payload.get("fetched_at")
        for game in payload.get("games") or []:
            name = str(game.get("name") or "")
            if " @ " in name:
                away_name, home_name = [p.strip() for p in name.split(" @ ", 1)]
            else:
                away_name, home_name = name, ""
            for prop in game.get("props") or []:
                if str(prop.get("stat") or "").lower() != "strikeouts":
                    continue
                over = prop.get("over") or {}
                under = prop.get("under") or {}
                rows.append(
                    {
                        "book": book,
                        "path": str(path.name),
                        "fetched_at": fetched,
                        "event_id": game.get("event_id"),
                        "game_name": name,
                        "away_name": away_name,
                        "home_name": home_name,
                        "scheduled": game.get("scheduled"),
                        "player": prop.get("player"),
                        "line": prop.get("line"),
                        "over_american": (over or {}).get("american"),
                        "under_american": (under or {}).get("american"),
                        "is_main": bool(prop.get("is_main")),
                        "sub_type": prop.get("sub_type"),
                    }
                )
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise ValueError("no pitcher strikeout props in odds files")
    return _finish_quotes(frame)


def _pair_over_under(raw: pd.DataFrame, book: str, table: str) -> pd.DataFrame:
    if raw.empty:
        return pd.DataFrame()
    frame = raw.copy()
    frame["side"] = frame["side"].astype(str).str.lower()
    frame["player"] = frame["player_name"].astype(str)
    frame["line"] = pd.to_numeric(frame["line_score"], errors="coerce")
    keys = ["player", "line"]
    meta = [c for c in ("event_id", "away_team", "home_team", "start_time", "sub_type", "is_main", "fetched_at") if c in frame.columns]
    base = frame.drop_duplicates(keys, keep="last")[keys + meta]
    over = (
        frame.loc[frame["side"].eq("over"), keys + ["american_price"]]
        .drop_duplicates(keys, keep="last")
        .rename(columns={"american_price": "over_american"})
    )
    under = (
        frame.loc[frame["side"].eq("under"), keys + ["american_price"]]
        .drop_duplicates(keys, keep="last")
        .rename(columns={"american_price": "under_american"})
    )
    paired = base.merge(over, on=keys, how="left").merge(under, on=keys, how="left")
    away = paired["away_team"] if "away_team" in paired.columns else ""
    home = paired["home_team"] if "home_team" in paired.columns else ""
    scheduled = paired["start_time"] if "start_time" in paired.columns else pd.NaT
    game_name = (
        (away.fillna("").astype(str) + " @ " + home.fillna("").astype(str)).str.strip(" @")
        if "away_team" in paired.columns
        else ""
    )
    return pd.DataFrame(
        {
            "book": book,
            "path": f"odds.{table}",
            "fetched_at": paired["fetched_at"] if "fetched_at" in paired.columns else pd.NaT,
            "event_id": paired["event_id"] if "event_id" in paired.columns else pd.NA,
            "game_name": game_name,
            "away_name": away,
            "home_name": home,
            "scheduled": scheduled,
            "player": paired["player"],
            "line": paired["line"],
            "over_american": paired["over_american"],
            "under_american": paired["under_american"],
            "is_main": paired["is_main"] if "is_main" in paired.columns else False,
            "sub_type": paired["sub_type"] if "sub_type" in paired.columns else "",
        }
    )


def _dfs_quotes(raw: pd.DataFrame, book: str, table: str) -> pd.DataFrame:
    if raw.empty:
        return pd.DataFrame()
    odds_type = raw["odds_type"].astype(str) if "odds_type" in raw.columns else "standard"
    # PrizePicks posts no price. A 2-pick Power Play pays 3x, so each leg breaks even at -137.
    standard = odds_type.str.lower().eq("standard")
    american = pd.Series(np.where(standard, -137, np.nan), index=raw.index, dtype="float")
    return pd.DataFrame(
        {
            "book": book,
            "path": f"odds.{table}",
            "fetched_at": raw["fetched_at"] if "fetched_at" in raw.columns else pd.NaT,
            "event_id": pd.NA,
            "game_name": "",
            "away_name": "",
            "home_name": "",
            "scheduled": pd.NaT,
            "player": raw["player_name"].astype(str),
            "line": pd.to_numeric(raw["line_score"], errors="coerce"),
            "over_american": american,
            "under_american": american.copy(),
            "is_main": standard,
            "sub_type": odds_type,
        }
    )


def load_latest_supabase_quotes() -> pd.DataFrame:
    _load_env(ROOT / ".env")
    url = os.environ.get("SUPABASE_DB_URL", "").strip()
    if not url:
        raise RuntimeError("SUPABASE_DB_URL missing — set it in .env")
    engine = create_engine(url, connect_args={"connect_timeout": 20, "sslmode": "require"})
    frames: list[pd.DataFrame] = []
    fetched: dict[str, object] = {}
    with engine.connect() as conn:
        for book, table, where, kind in ODDS_TABLES:
            if kind == "dfs":
                # Unchanged standard lines keep an older fetched_at. The newest
                # write is often only demon/goblin, so take the latest standard
                # line per pitcher from the last 36 hours.
                sql = text(
                    f"SELECT DISTINCT ON (player_name) * FROM odds.{table} "
                    f"WHERE {where} AND lower(odds_type) = 'standard' "
                    f"AND fetched_at >= (SELECT MAX(fetched_at) FROM odds.{table}) - INTERVAL '36 hours' "
                    f"ORDER BY player_name, fetched_at DESC"
                )
            else:
                sql = text(
                    f"SELECT * FROM odds.{table} "
                    f"WHERE fetched_at = (SELECT MAX(fetched_at) FROM odds.{table}) "
                    f"AND {where}"
                )
            raw = pd.read_sql(sql, conn)
            fetched[book] = None if raw.empty else raw["fetched_at"].max()
            if kind == "dfs":
                frames.append(_dfs_quotes(raw, book, table))
            else:
                frames.append(_pair_over_under(raw, book, table))
            print(f"{book:10} latest={fetched[book]} pitcher_k_rows={0 if frames[-1].empty else len(frames[-1])}")
    quotes = _finish_quotes(pd.concat(frames, ignore_index=True))
    if quotes.empty:
        raise ValueError("no pitcher strikeout props in latest Supabase snapshots")
    return quotes


if ODDS_SOURCE == "json":
    quotes = load_prop_files(ROOT, ODDS_GLOB)
else:
    quotes = load_latest_supabase_quotes()
print(quotes.groupby("book").size().rename("k_quotes").to_string())
print("games", quotes["game_name"].mask(quotes["game_name"].eq("")).nunique(), "players", quotes["player"].nunique())
quotes.head()


novig      latest=2026-09-25 18:39:15.876240+00:00 pitcher_k_rows=75
prophetx   latest=None pitcher_k_rows=0
pinnacle   latest=2026-09-25 18:40:58.258168+00:00 pitcher_k_rows=22
fanduel    latest=2026-09-25 18:40:16.809310+00:00 pitcher_k_rows=23
draftkings latest=2026-09-25 18:40:35.522357+00:00 pitcher_k_rows=176
underdog   latest=2026-09-25 18:39:50.937371+00:00 pitcher_k_rows=25
prizepicks latest=2026-09-25 18:41:19.634299+00:00 pitcher_k_rows=37
book
draftkings    176
fanduel        23
novig          75
pinnacle       22
prizepicks     37
underdog       25
games 14 players 41


,book,path,fetched_at,event_id,game_name,away_name,home_name,scheduled,player,line,over_american,under_american,is_main,sub_type,game_date,player_key,home_key,away_key
0,novig,odds.mlb_novig,2026-09-25 18:39:15.876240+00:00,01a0d5c4-a5cf-7200-8d96-e7678344dbcc,Baltimore Orioles @ New York Yankees,Baltimore Orioles,New York Yankees,2026-09-25 20:05:00+00:00,Brendan Beck,3.500,-388.000,125.000,False,pitcher_strikeouts,2026-09-25,brendan beck,new york yankees,baltimore orioles
1,novig,odds.mlb_novig,2026-09-25 18:39:15.876240+00:00,01a0d5c4-a5cf-7200-8d96-e7678344dbcc,Baltimore Orioles @ New York Yankees,Baltimore Orioles,New York Yankees,2026-09-25 20:05:00+00:00,Brendan Beck,5.500,128.000,-809.000,False,pitcher_strikeouts,2026-09-25,brendan beck,new york yankees,baltimore orioles
2,novig,odds.mlb_novig,2026-09-25 18:39:15.876240+00:00,01a0d5c4-a5cf-7200-8d96-e7678344dbcc,Baltimore Orioles @ New York Yankees,Baltimore Orioles,New York Yankees,2026-09-25 20:05:00+00:00,Trevor Rogers,4.500,-545.000,239.000,False,pitcher_strikeouts,2026-09-25,trevor rogers,new york yankees,baltimore orioles
3,novig,odds.mlb_novig,2026-09-25 18:39:15.876240+00:00,01a0d5c4-a5cf-7200-8d96-e7678344dbcc,Baltimore Orioles @ New York Yankees,Baltimore Orioles,New York Yankees,2026-09-25 20:05:00+00:00,Trevor Rogers,6.500,115.000,-125.000,True,pitcher_strikeouts,2026-09-25,trevor rogers,new york yankees,baltimore orioles
4,novig,odds.mlb_novig,2026-09-25 18:39:15.876240+00:00,01a0d5c4-a5cf-7200-8d96-e7678344dbcc,Baltimore Orioles @ New York Yankees,Baltimore Orioles,New York Yankees,2026-09-25 20:05:00+00:00,Trevor Rogers,5.500,-239.000,115.000,False,pitcher_strikeouts,2026-09-25,trevor rogers,new york yankees,baltimore orioles


## Today's schedule → `game_pk` / probable pitchers


In [48]:
def fetch_schedule_named(client, start: str, end: str) -> tuple[bytes, pd.DataFrame]:
    payload = client(
        SCHEDULE_URL,
        {
            "sportId": 1,
            "startDate": start,
            "endDate": end,
            "hydrate": "probablePitcher,team,venue",
        },
    )
    data = json.loads(payload)
    rows: list[dict] = []
    for day in data.get("dates") or []:
        for game in day.get("games") or []:
            teams = game.get("teams") or {}
            home = teams.get("home") or {}
            away = teams.get("away") or {}
            home_team = home.get("team") or {}
            away_team = away.get("team") or {}
            home_p = home.get("probablePitcher") or {}
            away_p = away.get("probablePitcher") or {}
            rows.append(
                {
                    "game_pk": int(game["gamePk"]),
                    "scheduled_start_utc": game.get("gameDate"),
                    "official_date": game.get("officialDate") or day.get("date"),
                    "status": (game.get("status") or {}).get("detailedState"),
                    "home_team_id": home_team.get("id"),
                    "away_team_id": away_team.get("id"),
                    "home_name": home_team.get("name"),
                    "away_name": away_team.get("name"),
                    "venue_id": (game.get("venue") or {}).get("id"),
                    "doubleheader": int(str(game.get("doubleHeader") or "N") != "N"),
                    "probable_home_pitcher_id": home_p.get("id"),
                    "probable_away_pitcher_id": away_p.get("id"),
                    "probable_home_pitcher": home_p.get("fullName"),
                    "probable_away_pitcher": away_p.get("fullName"),
                }
            )
    slate = pd.DataFrame(rows)
    if not slate.empty:
        slate["scheduled_start_utc"] = pd.to_datetime(slate["scheduled_start_utc"], utc=True)
        slate["home_key"] = slate["home_name"].map(normalize_player_name)
        slate["away_key"] = slate["away_name"].map(normalize_player_name)
        slate["home_p_key"] = slate["probable_home_pitcher"].map(normalize_player_name)
        slate["away_p_key"] = slate["probable_away_pitcher"].map(normalize_player_name)
    return payload, slate


slate_dates = sorted(
    {
        str(x)
        for x in quotes["game_date"].dropna().unique()
        if str(x) not in {"", "nan", "NaT"}
    }
)
today = pd.Timestamp(datetime.now(UTC)).normalize()
window = {today.strftime("%Y-%m-%d"), (today + pd.Timedelta(days=1)).strftime("%Y-%m-%d")}
slate_dates = sorted(set(slate_dates) | window)
print("quote/slate dates", slate_dates)
if http is None:
    raise RuntimeError("set FETCH = True once to pull today's schedule")
today_payload, slate = fetch_schedule_named(http, slate_dates[0], slate_dates[-1])
ingested_at = datetime.now(UTC)
snapshot_id = snapshot_raw(
    config,
    "schedule",
    {"startDate": slate_dates[0], "endDate": slate_dates[-1], "sportId": 1},
    today_payload,
    ingested_at,
)
versions = parse_schedule(today_payload, snapshot_id, ingested_at)
existing_gv = store.read_table("game_versions")
store.write_table(
    "game_versions",
    coerce_frame(pd.concat([existing_gv, versions], ignore_index=True), GAME_VERSION_COLUMNS),
)
print(slate[["game_pk", "away_name", "home_name", "scheduled_start_utc", "probable_away_pitcher", "probable_home_pitcher"]].to_string(index=False))


quote/slate dates ['2026-09-25', '2026-09-26']
 game_pk            away_name             home_name       scheduled_start_utc probable_away_pitcher probable_home_pitcher
  824703         Chicago Cubs        Boston Red Sox 2026-09-25 17:05:00+00:00           Clay Holmes           Alec Gamboa
  824706         Chicago Cubs        Boston Red Sox 2026-09-25 21:35:00+00:00        David Peterson          Brayan Bello
  823491    Baltimore Orioles      New York Yankees 2026-09-25 20:05:00+00:00         Trevor Rogers          Brendan Beck
  823489    Baltimore Orioles      New York Yankees 2026-09-25 20:10:00+00:00         Brandon Young                   NaN
  824220   Pittsburgh Pirates        Detroit Tigers 2026-09-25 22:40:00+00:00        Bubba Chandler          Jackson Jobe
  823409       Tampa Bay Rays Philadelphia Phillies 2026-09-25 22:40:00+00:00        Freddy Peralta    Cristopher Sánchez
  822681        New York Mets  Washington Nationals 2026-09-25 22:45:00+00:00                   NaN

In [49]:
SLATE_COLS = [
    "game_pk",
    "official_date",
    "home_key",
    "away_key",
    "home_team_id",
    "away_team_id",
    "venue_id",
    "doubleheader",
    "scheduled_start_utc",
    "probable_home_pitcher_id",
    "probable_away_pitcher_id",
    "home_p_key",
    "away_p_key",
    "home_name",
    "away_name",
]
dim = slate[SLATE_COLS]
have_teams = quotes["home_key"].fillna("").ne("") & quotes["away_key"].fillna("").ne("")
# Slate also has home_name/away_name; keep quote columns so leftover can reselect them.
by_game = quotes.loc[have_teams].merge(
    dim.drop(columns=["home_name", "away_name"]),
    left_on=["game_date", "home_key", "away_key"],
    right_on=["official_date", "home_key", "away_key"],
    how="left",
)
quote_cols = list(quotes.columns)
leftover = pd.concat(
    [
        quotes.loc[~have_teams],
        by_game.loc[by_game["game_pk"].isna(), quote_cols],
    ],
    ignore_index=True,
)
by_game = by_game.loc[by_game["game_pk"].notna()].copy()
by_pitcher = pd.concat(
    [
        dim.assign(player_key=dim["home_p_key"]),
        dim.assign(player_key=dim["away_p_key"]),
    ],
    ignore_index=True,
)
by_pitcher = by_pitcher.loc[by_pitcher["player_key"].fillna("").ne("")].drop_duplicates("player_key")
named = leftover.drop(
    columns=[c for c in leftover.columns if c in by_pitcher.columns and c != "player_key"]
).merge(by_pitcher, on="player_key", how="left")
if "game_name" in named.columns:
    blank = named["game_name"].fillna("").eq("") & named["away_name"].notna()
    named.loc[blank, "game_name"] = named.loc[blank, "away_name"].astype(str) + " @ " + named.loc[blank, "home_name"].astype(str)
matched = pd.concat([by_game, named], ignore_index=True)
matched["pitcher_id"] = np.where(
    matched["player_key"] == matched["home_p_key"],
    matched["probable_home_pitcher_id"],
    np.where(
        matched["player_key"] == matched["away_p_key"],
        matched["probable_away_pitcher_id"],
        pd.NA,
    ),
)
people_now = load_or_fetch_people(config, http or default_http(config), seasons=(2026,))
key_to_id = {
    str(row.player_key): int(row.mlb_id)
    for row in people_now.itertuples(index=False)
    if getattr(row, "player_key", "")
}
missing = matched["pitcher_id"].isna()
matched.loc[missing, "pitcher_id"] = matched.loc[missing, "player_key"].map(key_to_id)
matched["is_home"] = (matched["player_key"] == matched["home_p_key"]).astype("int64")
matched["team_id"] = np.where(matched["is_home"] == 1, matched["home_team_id"], matched["away_team_id"])
matched["opponent_team_id"] = np.where(
    matched["is_home"] == 1, matched["away_team_id"], matched["home_team_id"]
)
print("unmatched pitchers")
print(matched.loc[matched["game_pk"].isna() | matched["pitcher_id"].isna(), ["book", "player", "game_name", "line"]].drop_duplicates())
print(matched.groupby("book").size().rename("rows").to_string())
k_card = matched.dropna(subset=["game_pk", "pitcher_id"]).copy()
k_card["pitcher_id"] = k_card["pitcher_id"].astype(int)
k_card["game_pk"] = k_card["game_pk"].astype(int)
print("matched K quotes", len(k_card), "starters", k_card[["game_pk", "pitcher_id"]].drop_duplicates().shape[0])
print(k_card.groupby("book").size().rename("matched").to_string())


unmatched pitchers
           book             player game_name  line
324  prizepicks     Andrew Painter           5.000
325  prizepicks       Brady Singer           3.500
328  prizepicks          Bryan Woo           7.000
331  prizepicks     Cam Schlittler           6.000
333  prizepicks      Daniel Espino           1.500
338  prizepicks  Grayson Rodriguez           5.000
340  prizepicks        Ian Seymour           6.000
345  prizepicks      Mason Barnett           3.500
348  prizepicks       Nick Pivetta           4.500
350  prizepicks      Peter Lambert           4.000
351  prizepicks      Ranger Suarez           4.000
358  prizepicks      Tyler Glasnow           7.000
359  prizepicks        Tyler Mahle           6.000
book
draftkings    176
fanduel        23
novig          83
pinnacle       22
prizepicks     37
underdog       25
matched K quotes 353 starters 34
book
draftkings    176
fanduel        23
novig          83
pinnacle       22
prizepicks     24
underdog       25


## Phase 1 — PBP cache (365d) then today's live nines

Re-run is cheap: games already in `batter_pas` are skipped. Set `FETCH = False` after the first success if you want a fully offline phase 2.


In [50]:
today = pd.Timestamp(datetime.now(UTC)).normalize()
pbp_start = (today - pd.Timedelta(days=int(PBP_LOOKBACK_DAYS))).strftime("%Y-%m-%d")
pbp_end = today.strftime("%Y-%m-%d")
range_payload = http(
    SCHEDULE_URL,
    {"sportId": 1, "startDate": pbp_start, "endDate": pbp_end},
)
range_versions = parse_schedule(range_payload, "live-card-pbp-window", datetime.now(UTC))
range_pks = sorted(pd.to_numeric(range_versions["game_pk"], errors="coerce").dropna().astype(int).unique())
print(f"schedule {pbp_start}..{pbp_end}: {len(range_pks)} games")

pas = store.read_table("batter_pas")
have = set()
if not pas.empty:
    have = set(pd.to_numeric(pas["game_pk"], errors="coerce").dropna().astype(int))
gzip_dir = config.raw_dir / "mlb_pbp"
local_missing: list[int] = []
fetch_missing: list[int] = []
for pk in range_pks:
    if pk in have:
        continue
    if (gzip_dir / f"{pk}.json.gz").exists():
        local_missing.append(pk)
    else:
        fetch_missing.append(pk)
if PBP_LIMIT is not None:
    fetch_missing = fetch_missing[: int(PBP_LIMIT)]
print("already in batter_pas", len(have))
print("parse local gzip", len(local_missing), "fetch", len(fetch_missing))

id_map = store.read_table("id_map")
if id_map.empty:
    id_map = people_now.rename(columns={"mlb_id": "mlb_id"})

if local_missing:
    ingest_play_by_play(config, game_pks=local_missing, http=None, people=id_map)
    print("parsed local gzip", len(local_missing))
if fetch_missing and http is not None:
    chunk = 25
    for i in range(0, len(fetch_missing), chunk):
        batch = fetch_missing[i : i + chunk]
        ingest_play_by_play(config, game_pks=batch, http=http, people=id_map)
        print(f"pbp {i + len(batch)}/{len(fetch_missing)}")
elif fetch_missing:
    print("FETCH is False; leaving", len(fetch_missing), "games unfetched")

slate_pks = sorted(k_card["game_pk"].unique().tolist())
print("snapshot live nines", slate_pks)
if http is None:
    live_slots = ingest_lineup_slots(config, game_pks=slate_pks, provenance="live_feed", http=None)
else:
    live_slots = ingest_lineup_slots(config, game_pks=slate_pks, provenance="live_feed", http=http)
print("new lineup_slots rows", 0 if live_slots is None else len(live_slots))


schedule 2025-09-25..2026-09-25: 2996 games
already in batter_pas 9855
parse local gzip 56 fetch 0
parsed local gzip 56
snapshot live nines [822681, 822760, 823083, 823085, 823165, 823167, 823245, 823248, 823409, 823489, 823491, 823652, 823653, 823735, 823816, 824058, 824220, 824544, 824706, 824947]
new lineup_slots rows 0


## Today's nines


In [51]:
slots = store.read_table("lineup_slots")
cutoffs = (
    k_card[["game_pk", "scheduled_start_utc"]]
    .drop_duplicates("game_pk")
    .assign(
        cutoff=lambda d: pd.to_datetime(d["scheduled_start_utc"], utc=True)
        - pd.Timedelta(hours=float(config.forecast_horizon_hours))
    )
)
as_of_parts = []
rv = rate_version(config)
for row in cutoffs.itertuples(index=False):
    part = select_lineup_slots(slots, cutoff=row.cutoff, rate_version=rv)
    part = part.loc[part["game_pk"] == int(row.game_pk)]
    as_of_parts.append(part)
nines = pd.concat(as_of_parts, ignore_index=True) if as_of_parts else slots.iloc[0:0]
id_lookup = people_now.set_index("mlb_id")["name"] if "mlb_id" in people_now.columns else pd.Series(dtype=str)
if not nines.empty:
    nines["batter"] = nines["batter_id"].map(id_lookup).fillna(nines["batter_id"].astype(str))
show_cols = [
    c
    for c in (
        "game_pk",
        "side",
        "slot",
        "batter",
        "batter_id",
        "slot_is_pitcher",
        "k_pa_vs_hand_shrunk_365",
        "k_pa_overall_shrunk_365",
        "pa_vs_hand_365",
        "opposing_pitcher_hand",
        "observed_before_cutoff",
        "provenance",
    )
    if c in nines.columns
]
nines.sort_values(["game_pk", "side", "slot"])[show_cols]


,game_pk,side,slot,batter_id,slot_is_pitcher,k_pa_vs_hand_shrunk_365,k_pa_overall_shrunk_365,pa_vs_hand_365,opposing_pitcher_hand,observed_before_cutoff,provenance


## Log5 sketch (not wired)

\[
p_i = \mathrm{Log5}\big(K/\mathrm{BF}_{\text{pitcher}},\; K/\mathrm{PA}_{\text{slot }i},\; \text{league }K/\mathrm{PA}\big)
\]

Mixture placeholder: 100% mass on `expected_bf` from `nb_k_v1`. Walk the **opposing** nine in order, cycling after slot 9, for \(N=\max(1,\mathrm{round}(\mathrm{expected\_bf}))\) batters faced. \(\mathbb{E}[K]=\sum p_i\). Not Poisson-binomial.


In [52]:
def _clip_p(p: float) -> float:
    return float(min(max(p, 1e-6), 1.0 - 1e-6))


def log5(p_bat: float, p_pit: float, p_lg: float) -> float:
    a, b, lg = _clip_p(p_bat), _clip_p(p_pit), _clip_p(p_lg)
    num = (a * b) / lg
    den = num + ((1.0 - a) * (1.0 - b) / (1.0 - lg))
    return float(num / den)


pas = store.read_table("batter_pas")
pas_times = pd.to_datetime(pas["event_time_utc"], utc=True) if not pas.empty else pd.Series(dtype="datetime64[ns, UTC]")


def league_k_pa(cutoff: pd.Timestamp) -> float:
    if pas.empty:
        return 0.22
    window = pas.loc[
        (pas_times < cutoff)
        & (pas_times >= cutoff - pd.Timedelta(days=365))
        & (pas["event_time_imputed"] == 0)
        & (pas["is_pitcher_in_game"] == 0)
    ]
    if window.empty:
        return 0.22
    return float(window["event_type"].map(is_strikeout).mean())


def walk_expected_k(
    game_pk: int,
    pitcher_id: int,
    is_home: int,
    expected_bf: float,
    pitcher_k_bf: float,
    cutoff: pd.Timestamp,
) -> tuple[float, pd.DataFrame]:
    side = "away" if int(is_home) == 1 else "home"
    nine = nines.loc[(nines["game_pk"] == int(game_pk)) & (nines["side"] == side)].copy()
    nine = nine.sort_values("slot")
    lg = league_k_pa(cutoff)
    n_bf = max(1, int(round(float(expected_bf))))
    if nine.empty:
        return float("nan"), nine
    rates = nine["k_pa_vs_hand_shrunk_365"].to_numpy(dtype=float)
    overall = nine["k_pa_overall_shrunk_365"].to_numpy(dtype=float)
    rates = np.where(np.isfinite(rates), rates, overall)
    ps = [log5(float(rates[i % 9]), float(pitcher_k_bf), lg) for i in range(n_bf)]
    walked = nine.copy()
    walked["log5_p"] = [log5(float(rates[i]), float(pitcher_k_bf), lg) for i in range(len(rates))]
    return float(np.sum(ps)), walked


## `nb_k_v1` as-is + card


In [53]:
people_hist = load_or_fetch_people(
    config, http or default_http(config), seasons=TRAIN_SEASONS
)
hist = load_or_fetch_starter_logs(
    config, http or default_http(config), people_hist, seasons=TRAIN_SEASONS
)
schedule_hist = load_or_fetch_schedule(
    config, http or default_http(config), seasons=TRAIN_SEASONS
)
hist = attach_schedule_times(hist, schedule_hist)
throws = people_now.set_index("mlb_id")["throws"] if "throws" in people_now.columns else pd.Series(dtype=str)
now = pd.Timestamp(datetime.now(UTC))
live_starts = (
    k_card.drop_duplicates(["game_pk", "pitcher_id"])
    .assign(
        game_date=lambda d: pd.to_datetime(d["scheduled_start_utc"], utc=True).dt.strftime("%Y-%m-%d"),
        season=lambda d: pd.to_datetime(d["scheduled_start_utc"], utc=True).dt.year.astype(int),
        strikeouts=0,
        batters_faced=0,
        pitches=0,
        outs=0,
        role="starter",
        pitcher_hand=lambda d: d["pitcher_id"].map(throws).fillna(""),
        event_time_utc=lambda d: pd.to_datetime(d["scheduled_start_utc"], utc=True),
        ingested_at_utc=now,
    )
)
keep = [
    c
    for c in (*PITCHER_START_COLUMNS, "player_key")
    if c in hist.columns or c in live_starts.columns
]
starts = pd.concat(
    [hist.reindex(columns=keep), live_starts.reindex(columns=keep)],
    ignore_index=True,
)
pregame = pregame_from_starts(starts, config)
tables = {
    "pitcher_starts": starts,
    "pregame_snapshots": pregame,
    "pitch_events": pd.DataFrame(),
    "plate_appearances": pd.DataFrame(),
    "game_versions": store.read_table("game_versions"),
    "id_map": id_map if not id_map.empty else people_now,
}
feature_rows = _model_feature_rows(tables, config)
panel = _evaluation_panel(tables, feature_rows, config)
target_keys = k_card[["pitcher_id", "game_pk"]].drop_duplicates()
target = panel.merge(target_keys, on=["pitcher_id", "game_pk"], how="inner")
if target.empty:
    raise ValueError("no feature rows for today's matched starters — check starter-log coverage")
preds = predict_strikeout_pmf(model, target, config)
print(preds[["pitcher_id", "game_pk", "expected_k", "expected_bf"]].to_string(index=False))


GLM optimizer did not converge after 17 iterations; fold=? intercept_only=False method=glm
GLM optimizer did not converge after 18 iterations; fold=? intercept_only=False method=glm
GLM optimizer did not converge after 55 iterations; fold=? intercept_only=False method=glm
GLM optimizer did not converge after 100 iterations; fold=? intercept_only=False method=glm
GLM optimizer did not converge after 52 iterations; fold=? intercept_only=False method=glm
GLM optimizer did not converge after 74 iterations; fold=? intercept_only=False method=glm
GLM optimizer did not converge after 74 iterations; fold=? intercept_only=False method=glm
GLM optimizer did not converge after 69 iterations; fold=? intercept_only=False method=glm
GLM optimizer did not converge after 55 iterations; fold=? intercept_only=False method=glm
GLM optimizer did not converge after 48 iterations; fold=? intercept_only=False method=glm
GLM optimizer did not converge after 102 iterations; fold=? intercept_only=False method=g

 pitcher_id  game_pk  expected_k  expected_bf
     694341   823491       3.520       22.443
     694341   823489       1.915        8.849
     669432   823491       3.991       21.980
     669432   823489       4.170       19.542
     642547   823409       3.904       20.363
     650911   823409       5.219       23.275
     695549   824220       4.421       22.798
     696149   824220       3.740       19.336
     674841   822681       3.976       20.555
     666157   822760       3.912       21.306
     691587   823816       5.120       21.230
     688107   823735       3.782       19.938
     700241   823735       3.718       21.396
     608372   824544       3.329       18.725
     680732   824544       3.821       19.702
     668909   824058       4.454       24.190
     702070   824058       4.707       23.357
     594798   823653       4.440       20.569
     657746   823653       4.442       17.602
     694297   823245       4.642       21.057
     672282   823083       4.492  

In [54]:
def _quote_evenness(over_american, under_american) -> float:
    """Distance from a 50/50 two-way. A missing side counts as fully one-sided."""
    score = 0.0
    for price in (over_american, under_american):
        if pd.isna(price):
            score += 0.5
        else:
            score += abs(float(american_to_implied(float(price))) - 0.5)
    return score


def _main_lines(frame: pd.DataFrame) -> pd.DataFrame:
    """One strikeout quote per book and pitcher.

    Books that stamp ``is_main`` (Novig, ProphetX, PrizePicks standard) keep that row.
    The rest keep the line closest to even money, which is the main rung on a
    one-sided DraftKings ladder.
    """
    if frame.empty:
        return frame
    flagged = (
        frame["is_main"].fillna(False).astype(bool)
        if "is_main" in frame.columns
        else pd.Series(False, index=frame.index)
    )
    books_with_main = set(frame.loc[flagged, "book"].astype(str))
    marked = frame.loc[frame["book"].astype(str).isin(books_with_main) & flagged]
    unmarked = frame.loc[~frame["book"].astype(str).isin(books_with_main)].copy()
    if not unmarked.empty:
        unmarked["_even"] = [
            _quote_evenness(over, under)
            for over, under in zip(unmarked["over_american"], unmarked["under_american"], strict=True)
        ]
        unmarked = (
            unmarked.sort_values(["_even", "line"])
            .drop_duplicates(["book", "player"], keep="first")
            .drop(columns="_even")
        )
    return pd.concat([marked, unmarked], ignore_index=True)


pmf = preds.loc[:, list(PMF_COLUMNS)].to_numpy(dtype=float)
k_max = pmf.shape[1] - 2
support = np.concatenate([np.arange(k_max + 1, dtype=float), np.array([float(k_max + 1)])])

main_card = _main_lines(k_card)
print(f"main lines {len(main_card)} of {len(k_card)}")

card_rows = []
walks = []
for rec in main_card.itertuples(index=False):
    hit = preds.loc[
        (preds["pitcher_id"] == int(rec.pitcher_id)) & (preds["game_pk"] == int(rec.game_pk))
    ]
    if hit.empty:
        continue
    i = hit.index[0]
    loc = list(preds.index).index(i)
    line = float(rec.line)
    p_over = float(pmf[loc] @ (support > line).astype(float))
    p_under = float(pmf[loc] @ (support < line).astype(float))
    p_push = float(pmf[loc] @ np.isclose(support, line).astype(float))
    exp_k = float(hit["expected_k"].iloc[0])
    exp_bf = float(hit["expected_bf"].iloc[0]) if pd.notna(hit["expected_bf"].iloc[0]) else float("nan")
    k_bf = float(target.loc[target.index == i, "k_bf_shrunk_365"].iloc[0]) if "k_bf_shrunk_365" in target.columns else 0.22
    cutoff = pd.to_datetime(rec.scheduled_start_utc, utc=True) - pd.Timedelta(
        hours=float(config.forecast_horizon_hours)
    )
    sketch_k, walked = walk_expected_k(
        int(rec.game_pk), int(rec.pitcher_id), int(rec.is_home), exp_bf, k_bf, cutoff
    )
    walked = walked.assign(player=rec.player, book=rec.book)
    walks.append(walked)
    over_imp = (
        float(american_to_implied(float(rec.over_american)))
        if pd.notna(rec.over_american)
        else float("nan")
    )
    under_imp = (
        float(american_to_implied(float(rec.under_american)))
        if pd.notna(rec.under_american)
        else float("nan")
    )
    ev_over = (
        expected_profit(p_over, p_under, implied_to_decimal(over_imp), p_push=p_push)
        if np.isfinite(over_imp)
        else float("nan")
    )
    ev_under = (
        expected_profit(p_under, p_over, implied_to_decimal(under_imp), p_push=p_push)
        if np.isfinite(under_imp)
        else float("nan")
    )
    card_rows.append(
        {
            "book": rec.book,
            "player": rec.player,
            "game": rec.game_name,
            "game_pk": int(rec.game_pk),
            "pitcher_id": int(rec.pitcher_id),
            "scheduled_start_utc": pd.Timestamp(rec.scheduled_start_utc).isoformat(),
            "line": line,
            "over": rec.over_american,
            "under": rec.under_american,
            "implied_over": over_imp,
            "implied_under": under_imp,
            "nb_k_v1_E[K]": exp_k,
            "log5_sketch_E[K]": sketch_k,
            "expected_bf": exp_bf,
            "k_bf_shrunk_365": k_bf,
            "p_over": p_over,
            "p_under": p_under,
            "ev_over": ev_over,
            "ev_under": ev_under,
        }
    )

card = pd.DataFrame(card_rows)
if not card.empty:
    card["nb_side"] = np.where(card["ev_over"] > card["ev_under"], "over", "under")
    card["nb_ev"] = card[["ev_over", "ev_under"]].max(axis=1)
print(card.sort_values("nb_ev", ascending=False).to_string(index=False))


main lines 143 of 353
      book             player                                       game  game_pk  pitcher_id       scheduled_start_utc  line     over    under  implied_over  implied_under  nb_k_v1_E[K]  log5_sketch_E[K]  expected_bf  k_bf_shrunk_365  p_over  p_under  ev_over  ev_under nb_side  nb_ev
     novig       Brendan Beck       Baltimore Orioles @ New York Yankees   823489      694341 2026-09-25T20:10:00+00:00 4.500 -146.000  135.000         0.593          0.426         1.915               NaN        8.849            0.220   0.049    0.951   -0.918     1.235   under  1.235
   fanduel         Sean Burke       Colorado Rockies @ Chicago White Sox   824544      680732 2026-09-25T23:40:00+00:00 5.500 -178.000  138.000         0.640          0.420         3.821               NaN       19.702            0.220   0.194    0.806   -0.696     0.917   under  0.917
     novig         Sean Burke       Colorado Rockies @ Chicago White Sox   824544      680732 2026-09-25T23:40:00+00:00 

In [55]:
snapshot_at = datetime.now(UTC)
stamp = snapshot_at.strftime("%Y-%m-%d_%H%M%S")
out = ROOT / "data" / "ev_snapshots" / "mlb" / f"strikeouts_{stamp}.csv"
out.parent.mkdir(parents=True, exist_ok=True)
saved = card.copy()
saved.insert(0, "snapshot_at_utc", snapshot_at.isoformat())
saved.to_csv(out, index=False)
print(out)
print("rows", len(saved), "positive", int((saved["nb_ev"] > 0).sum()))


/Users/alexgonzalez/Documents/nba_quant/data/ev_snapshots/mlb/strikeouts_2026-09-25_184245.csv
rows 143 positive 106
